In [1]:
# Skeleton Training Prototype
# ===========================
# - This is the jupyter notebook I created to define and test out the training process for Skeleton -> moved to lightningai studios
#     - a cross-attention transformer that predicts the next coordinate (3D vector) relative to a ligand centroid given its molecular fingerprint
# - You can run this notebooks's cells yourself, but just be aware the model that gets produced is watered down and its almost definitely going to output coordinates that oscillate around the same area.
#     - Compared to the cloud script, it has half the embedding size, much lower batch size and half the heads.
#     - still fun to mess around with though, I was able to get some prototypes going that could kind of grasp the general pattern
#     - also included some small tests on it to see if it was learning anything (checking output for different molecules, coordinate output statistics)

In [2]:
import torch
import pandas as pd
%matplotlib inline

from src.paths import *
from src.nano_maker.modules.data_handling.radial_dataset import RadialDataset
from src.nano_maker.skeleton import Skeleton

In [3]:
# Load data
# print(SMILE_2_PDBHITS.columns)  # 'SMILES', 'PDB_hits'
# print(MOLECULAR_FINGERPRINTS.columns)  # 'smiles_str', 'map4_fp'
# print(RADIAL_SEQUENCES.columns)  # 'PDB_ID', 'radial_sequence'
PROJECT_ROOT = Path.cwd().parent
DATAPATH = PROJECT_ROOT / "database"

RADIAL_SEQUENCES = pd.read_pickle(DATAPATH / "radial_seq_df.pkl")
RADIAL_SEQUENCES.set_index("PDB_ID", inplace=True)

MOLECULAR_FINGERPRINTS = pd.read_pickle(DATAPATH / "molfp_df.pkl")
MOLECULAR_FINGERPRINTS.set_index("smiles_str", inplace=True)

train_pointers_path = DATAPATH / "skeleton_training_pointers_dict.parquet"
test_pointers_path = DATAPATH / "skeleton_test_pointers_dict.parquet"

In [19]:
# RadialSeeker parameters used -> record these - might design a config file later
max_angstroms = 33

block_size = 75
batch_size = 64
n_embd = 256
n_head = 4
n_layers = 4
map4_res = 1024
dropout=0.1
device = 'cuda' if torch.cuda.is_available() else 'cpu'
loss_alpha = 0.5
n_epochs = 5

# training dataset
training_dataset = RadialDataset(pointer_path=train_pointers_path,
                                 smiles_molfp=MOLECULAR_FINGERPRINTS,
                                 pdb_radial=RADIAL_SEQUENCES,
                                 block_size=block_size,
                                 max_angstroms=max_angstroms)

# DataLoader
from torch.utils.data import DataLoader
loader = DataLoader(
    training_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    drop_last=True
)

torch.manual_seed(311104)

# test / validation set
test_set = RadialDataset(pointer_path=test_pointers_path,
                                 smiles_molfp=MOLECULAR_FINGERPRINTS,
                                 pdb_radial=RADIAL_SEQUENCES,
                                 block_size=block_size,
                                 max_angstroms=max_angstroms)
# DataLoader
from torch.utils.data import DataLoader
test_loader = DataLoader(
    test_set,
    batch_size=batch_size,
    num_workers=4,
    shuffle=False,
    drop_last=True
)

In [21]:
model = Skeleton(n_embd=n_embd, n_head=n_head, n_layers=n_layers,
                      block_size=block_size, map4_res=map4_res, max_angstroms=max_angstroms,
                      dropout=dropout).to(device)

In [22]:
learning_rate = 3e-4   # used 1e-4 on cloud
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
model.train()

SkeletonModel(
  (c_project): Linear(in_features=3, out_features=256, bias=True)
  (pos_emb): Embedding(75, 256)
  (mol_encoder): MolecularEncoder(
    (block1): Sequential(
      (0): Linear(in_features=1024, out_features=1024, bias=True)
      (1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.1, inplace=False)
    )
    (check1): Linear(in_features=1024, out_features=512, bias=True)
    (block2): Sequential(
      (0): Linear(in_features=512, out_features=512, bias=True)
      (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.1, inplace=False)
    )
    (check2): Linear(in_features=512, out_features=256, bias=True)
    (residual1): Linear(in_features=1024, out_features=512, bias=True)
    (block3): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (2): G

In [ ]:
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=1000
)

cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=len(loader) * n_epochs,
    eta_min=3e-5
)

lr_scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[1000]
)

In [23]:
@torch.no_grad()
def estimate_loss(model, loader, device, max_batches=None):
    model.eval()
    total_loss = 0
    n_batches = 0

    for batch_idx, batch in enumerate(loader):
        if max_batches and batch_idx >= max_batches:
            break

        map4_fp, radial_X, radial_Y = batch

        map4_fp = map4_fp.to(device)
        radial_X = radial_X.to(device)
        radial_Y = radial_Y.to(device)

        _, loss = model(radial_X, map4_fp, radial_Y)
        total_loss += loss.item()
        n_batches += 1

    model.train()
    return total_loss / n_batches


In [24]:
for epoch in range(n_epochs):   # start with few epochs
    total_loss = 0
    for batch_idx, batch in enumerate(loader):
        map4_fp, radial_X, radial_Y = batch

        map4_fp = map4_fp.to(device)
        radial_X = radial_X.to(device)
        radial_Y = radial_Y.to(device)

        optimizer.zero_grad()
        pred, loss = model(radial_X, map4_fp, radial_Y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f"Epoch {epoch + 1} | Batch {batch_idx + 1} | Loss: {loss.item():.6f}")

    train_loss = total_loss / len(loader)
    val_loss = estimate_loss(model, test_loader, device, max_batches=None)

    print(f"Epoch {epoch+1} | Train: {train_loss:.6f} | Val: {val_loss:.6f} | Gap: {val_loss - train_loss:.6f}")


Epoch 1 | Batch 1 | Loss: 47.097935
Epoch 1 | Batch 51 | Loss: 2.430671
Epoch 1 | Batch 101 | Loss: 0.983257
Epoch 1 | Batch 151 | Loss: 1.336504
Epoch 1 | Train: 2.756710 | Val: 1.068619 | Gap: -1.688091
Epoch 2 | Batch 1 | Loss: 1.559335
Epoch 2 | Batch 51 | Loss: 0.861494
Epoch 2 | Batch 101 | Loss: 0.985174
Epoch 2 | Batch 151 | Loss: 1.053876
Epoch 2 | Train: 0.991198 | Val: 0.937657 | Gap: -0.053541
Epoch 3 | Batch 1 | Loss: 0.857410
Epoch 3 | Batch 51 | Loss: 0.730391
Epoch 3 | Batch 101 | Loss: 0.911995
Epoch 3 | Batch 151 | Loss: 0.726044
Epoch 3 | Train: 1.005612 | Val: 0.956915 | Gap: -0.048697
Epoch 4 | Batch 1 | Loss: 0.905065
Epoch 4 | Batch 51 | Loss: 0.854258
Epoch 4 | Batch 101 | Loss: 0.803460
Epoch 4 | Batch 151 | Loss: 0.984386
Epoch 4 | Train: 0.921512 | Val: 0.928140 | Gap: 0.006628
Epoch 5 | Batch 1 | Loss: 1.026312
Epoch 5 | Batch 51 | Loss: 1.037518
Epoch 5 | Batch 101 | Loss: 0.938453
Epoch 5 | Batch 151 | Loss: 0.867394
Epoch 5 | Train: 0.927077 | Val: 0.8642

In [ ]:
# can halt training and save the partially-trained prototype here

torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': lr_scheduler.state_dict(),
}, CONTAINER / 'naanobot_prototypeI4_PRESCAFFOLD.pt')